In [41]:
import xarray as xr
import numpy as np

file = "D:/Repositories/too_much_big_data/" + "CHL_DATA/daily_chl_2024-08-21.nc"
chl = xr.open_dataset(file, group="geophysical_data")  # chlor_a
nav = xr.open_dataset(file, group="navigation_data")   # latitude, longitude

chlor = chl["chlor_a"].values
lat2d = nav["latitude"].values
lon2d = nav["longitude"].values


target_lat, target_lon = 33.3, 125.5  # 예: 제주도 남쪽

d = np.hypot(lat2d - target_lat, lon2d - target_lon)
iy, ix = np.unravel_index(np.nanargmin(d), d.shape)


val = chlor[iy, ix]
actual_lat = lat2d[iy, ix]
actual_lon = lon2d[iy, ix]
print(f"value={val:.4f} mg/m³ @ ({actual_lat:.4f}, {actual_lon:.4f})")

value=nan mg/m³ @ (33.2995, 125.5012)


In [47]:
chl["chlor_a"]

<xarray.DataArray 'chlor_a' (lat: 8000, lon: 10500)> Size: 336MB
array([[nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       ...,
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan]], dtype=float32)
Dimensions without coordinates: lat, lon
Attributes:
    units:      mg m^-3
    long_name:  Chlorophyll-a concentration

In [23]:
import xarray as xr
import dask
import os
from dask.diagnostics import ProgressBar
import re
import pandas as pd

dask.config.set(scheduler='processes')

In [7]:
data_dir = 'D:\\Repositories\\too_much_big_data\\CHL_DATA'
files = sorted([os.path.join(data_dir, f) for f in os.listdir(data_dir)])
files

['D:\\Repositories\\too_much_big_data\\CHL_DATA\\daily_chl_2021-11-03.nc',
 'D:\\Repositories\\too_much_big_data\\CHL_DATA\\daily_chl_2021-11-04.nc',
 'D:\\Repositories\\too_much_big_data\\CHL_DATA\\daily_chl_2021-11-05.nc',
 'D:\\Repositories\\too_much_big_data\\CHL_DATA\\daily_chl_2021-11-06.nc',
 'D:\\Repositories\\too_much_big_data\\CHL_DATA\\daily_chl_2021-11-07.nc',
 'D:\\Repositories\\too_much_big_data\\CHL_DATA\\daily_chl_2021-11-08.nc',
 'D:\\Repositories\\too_much_big_data\\CHL_DATA\\daily_chl_2021-11-09.nc',
 'D:\\Repositories\\too_much_big_data\\CHL_DATA\\daily_chl_2021-11-10.nc',
 'D:\\Repositories\\too_much_big_data\\CHL_DATA\\daily_chl_2021-11-11.nc',
 'D:\\Repositories\\too_much_big_data\\CHL_DATA\\daily_chl_2021-11-12.nc',
 'D:\\Repositories\\too_much_big_data\\CHL_DATA\\daily_chl_2021-11-13.nc',
 'D:\\Repositories\\too_much_big_data\\CHL_DATA\\daily_chl_2021-11-14.nc',
 'D:\\Repositories\\too_much_big_data\\CHL_DATA\\daily_chl_2021-11-15.nc',
 'D:\\Repositories\\too_m

In [8]:
def extract_time(f_name):
    match_ = re.search(r"(\d{4}-\d{2}-\d{2})", f_name)
    return match_.group(1) if match_ else None

extract_time(files[0])

'2021-11-03'

In [14]:
dataset = []
err_file = []
for f in files:
    time_date = extract_time(f)
    if time_date is None:
        continue
    try:
        ds = xr.open_dataset(f, chunks={}, engine='netcdf4')
        ds = ds.expand_dims(time=[time_date])
        dataset.append(ds)
    except:
        print(time_date)
        err_file.append(time_date)
err_file

2021-11-27


['2021-11-27']

In [34]:
ds_all = xr.concat(dataset, dim='time')

ds_all["time"] = pd.to_datetime(ds_all["time"].values)
monthly_ds = ds_all.resample(time='1M').mean()
# print(ds_all["time"].values)
print(monthly_ds)
monthly_ds

<xarray.Dataset> Size: 376B
Dimensions:  (time: 47)
Coordinates:
  * time     (time) datetime64[ns] 376B 2021-11-30 2021-12-31 ... 2025-09-30
Data variables:
    *empty*
Attributes:
    Conventions:       CF-1.6
    platform:          GK2B
    instrument:        GOCI-II
    processing_level:  L2
    title:             GOCI2 Daily Merged Data
    history:           merged from 8 files
    source_files:      GK2B_GOCI2_L2_20211103_011530_LA_Chl.nc, GK2B_GOCI2_L2...
    creation_date:     2025-09-29 20:37:05


c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\xarray\groupers.py:530: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  self.index_grouper = pd.Grouper(


<xarray.Dataset> Size: 376B
Dimensions:  (time: 47)
Coordinates:
  * time     (time) datetime64[ns] 376B 2021-11-30 2021-12-31 ... 2025-09-30
Data variables:
    *empty*
Attributes:
    Conventions:       CF-1.6
    platform:          GK2B
    instrument:        GOCI-II
    processing_level:  L2
    title:             GOCI2 Daily Merged Data
    history:           merged from 8 files
    source_files:      GK2B_GOCI2_L2_20211103_011530_LA_Chl.nc, GK2B_GOCI2_L2...
    creation_date:     2025-09-29 20:37:05